Description: Implementación y análisis del algoritmo SARSA Semi-gradiente episódico utilizando aproximación de funciones mediante Redes Neuronales Artificiales (PyTorch) para resolver el aterrizaje de un módulo lunar en LunarLander-v3.

In [1]:
# @title Instalación e importación de librerías
# !pip install "gymnasium[box2d]" torch numpy matplotlib tqdm
from agents import SarsaLunarAgent
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random

# Garantizamos reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ModuleNotFoundError: No module named 'tiles3'

Experimento

El experimento evalúa el rendimiento de un agente que utiliza el algoritmo on-policy SARSA Semi-gradiente para aprender a aterrizar el módulo lunar.Este entorno consta de:
Espacio de estados: Vector continuo de 8 dimensiones (coordenadas x/y, velocidades x/y, ángulo, velocidad angular, y contacto de las dos patas).
Espacio de acciones: 4 acciones discretas (0: nada, 1: motor izquierdo, 2: motor principal, 3: motor derecho).
Recompensa: Recompensas densas basadas en la distancia a la plataforma, velocidad, inclinación, uso de combustible y un premio/castigo final de +100/-100 por aterrizar o estrellarse. El entorno se considera "resuelto" si se alcanzan 200 puntos.

Objetivo del experimento:
Implementar la parametrización de la función de valor de acción $\hat{q}(s, a, \mathbf{w})$ mediante una red neuronal profunda.
Implementar el bucle de control de SARSA utilizando la propagación hacia atrás (backpropagation) para aproximar el gradiente $\nabla \hat{q}(S, A, \mathbf{w})$.
Analizar la curva de aprendizaje y la estabilidad del entrenamiento puro on-line sin buffer de repetición.

Arquitectura de la Red Neuronal y PolíticaA diferencia de los métodos tabulares, aquí los pesos $\mathbf{w}$ son los parámetros de la red neuronal.

In [ ]:

env = gym.make("LunarLander-v2")
agent = SarsaLunarAgent(env.action_space, env.observation_space)
episodes = 400
total_rewards = []

for i in range(episodes):
    state = env.reset()[0]
    action = agent.select_initial_action(state)
    total_reward = 0
    for t in range(1000):
        next_state, reward, done, truncated, info = env.step(action)
        action = agent.observe(state, action, reward, next_state, done)
        total_reward += reward
        state = next_state
        if done:
            break
    total_rewards.append(total_reward)
    print(f"Episode {i+1}/{episodes}, reward: {total_reward}")

env.close()
import matplotlib.pyplot as plt
plt.plot(total_rewards)
plt.xlabel("Episodes")
plt.ylabel("Total Reward")
plt.show()

In [ ]:
# @title Bucle de Entrenamiento SARSA Semi-gradiente

def train_sarsa_lunar_lander(episodes=1000):
    env = gym.make("LunarLander-v3")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    agent = SemiGradientSarsaAgent(state_dim, action_dim, lr=5e-4)
    
    rewards_history = []
    
    for ep in tqdm(range(episodes)):
        state, _ = env.reset()
        action = agent.get_action(state)  # A
        
        total_reward = 0
        done = False
        
        while not done:
            # Tomar acción A, observar R, S'
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Elegir A' desde S' usando la política actual
            next_action = agent.get_action(next_state)
            
            # Actualizar pesos: w <- w + alpha * [R + gamma*q(S',A') - q(S,A)] * grad(q)
            agent.update(state, action, reward, next_state, next_action, done)
            
            state = next_state
            action = next_action
            total_reward += reward
            
        agent.decay_epsilon()
        rewards_history.append(total_reward)
        
    env.close()
    return rewards_history, agent

# Ejecutar el entrenamiento
rewards, trained_agent = train_sarsa_lunar_lander(episodes=1500)

c:\Users\aaron\miniconda3\envs\dlpln\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
100%|██████████| 1500/1500 [19:35<00:00,  1.28it/s]


In [3]:
from tqdm import tqdm
import random
import numpy as np
import gymnasium as gym
from agents import LunarLanderTileCoding, LunarAgentSARSA

random.seed(42)
np.random.seed(42)

def train_agent_SARSA(agent, n_episodes, saving=False):
    saved_episodes = []
    n_save = max(1, n_episodes // 10)
    
    def save_episode(episode_num):
        return episode_num % n_save == 0
    
    env = agent.env

    for n in tqdm(range(n_episodes)):
        # Start a new episode (obs ya es una lista de features activas gracias al Wrapper)
        obs, info = env.reset()
        done = False
        episode_history = []

        # Play one complete episode
        action, is_exploring = agent.get_action(obs)                            
        
        while not done:
            next_obs, reward, terminated, truncated, info = env.step(action)        

            next_action, next_is_exploring = agent.get_action(next_obs)                  
            
            # Update Q-values (Semigradient SARSA)
            agent.update(obs, action, reward, terminated, next_obs, next_action)    

            # Move to next state
            done = terminated or truncated
            obs = next_obs
            action = next_action
            is_exploring = next_is_exploring

        # Reduce exploration rate (agent becomes less random over time)
        agent.decay_epsilon()
        
    if saving:
        return saved_episodes

# ----- CONFIGURACIÓN Y EJECUCIÓN -----

learning_rate, n_episodes = (0.2, 2000) # LunarLander puede necesitar bastantes episodios

# # Creamos el entorno original y lo envolvemos en nuestro Tile Coder
# base_env = gym.make("LunarLander-v3")
# # 4 bins por cada una de las 8 dimensiones
bins = np.array([4, 4, 4, 4, 4, 4, 2, 2])  # número de intervalos por dimensión
# env = LunarLanderTileCoding(base_env, bins=bins, n_tilings=8)

# # Utilizamos RecordEpisodeStatistics para el tracking (requiere acceder a env.unwrapped.return_queue)
# env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)

initial_epsilon = 1.0
final_epsilon = 0.05
ratio = 0.8 # Exploramos durante el 80% del entrenamiento

# agent = LunarAgentSARSA(
#     env=env,
#     learning_rate=learning_rate,
#     initial_epsilon=initial_epsilon,
#     epsilon_decay=(initial_epsilon - final_epsilon) / (n_episodes * ratio),
#     final_epsilon=final_epsilon,
#     decay_type="linear"
# )

# 1. Creamos el entorno base (v3)
base_env = gym.make("LunarLander-v3")

# 2. Aplicamos TU Tile Coding
# Suponiendo que bins y n_tilings están definidos
tile_env = LunarLanderTileCoding(base_env, bins=bins, n_tilings=8)

# 3. Guardamos los datos necesarios ANTES de envolverlo con estadísticas
n_tilings = tile_env.n_tilings
total_features = tile_env.total_features

# 4. Envolvemos para estadísticas
env = gym.wrappers.RecordEpisodeStatistics(tile_env, buffer_length=n_episodes)
env.total_features = n_tilings
env.n_tilings = total_features

# 5. Pasamos el entorno al agente
# El agente ahora podrá acceder a n_tilings si lo modificamos ligeramente 
# o si usamos las variables que ya extrajimos.
agent = LunarAgentSARSA(
    env=env,
    learning_rate=learning_rate,
    initial_epsilon=initial_epsilon,
    epsilon_decay=(initial_epsilon - final_epsilon) / (n_episodes * ratio),
    final_epsilon=final_epsilon,
    decay_type="linear"
)

saved_episodes = train_agent_SARSA(agent, n_episodes, saving=True)

# Evaluación rápida usando la cola de recompensas del wrapper
import matplotlib.pyplot as plt
returns = np.array(env.return_queue).flatten()
plt.plot(returns)
plt.title('Recompensas de LunarLander (Semigradiente SARSA)')
plt.xlabel('Episodios')
plt.ylabel('Recompensa Total')
plt.show()

print(f"Recompensa promedio de los últimos 100 episodios: {np.mean(returns[-100:])}")

# Nota: Tus funciones utils.plot_training_metrics y taxi_gif.animar_estados_taxi_gif
# deben ser reescritas para procesar imágenes continuas si quieres usarlas aquí.

  0%|          | 0/2000 [00:00<?, ?it/s]


IndexError: index 10903 is out of bounds for axis 0 with size 8